# Milestone 3: Transfer Learning für Galaxy10 DECals

In diesem Notebook wenden wir **Transfer Learning** auf etablierte reine CNN-Architekturen an (keine Transformer). 
Wir vergleichen drei verschiedene Backbones:
1. **ResNet50** (Residual Networks)
2. **InceptionV3** (GoogLeNet Architektur)
3. **EfficientNetB0** (Die Standard-Architektur für astronomische Zoobot-Modelle)

Das Notebook nutzt unsere bestehende K-Fold-Pipeline aus `train.py` und die Evaluierungstools aus `evaluate.py`.

In [1]:
import os
import glob
import tensorflow as tf
import numpy as np
from tensorflow.random import set_seed
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import ResNet50, InceptionV3, EfficientNetB0, ResNet152

# Importiere aus deinen lokalen Skripten
from data import load_data, prepare_splits, INPUT_SHAPE, NUM_CLASSES
from train import run_training
from evaluate import evaluate_models, plot_confusion_matrix, plot_training_history

print(f"TensorFlow Version: {tf.__version__}")
print(f"Input Shape: {INPUT_SHAPE}")
print(f"Anzahl Klassen: {NUM_CLASSES}")

2026-06-06 14:56:35.031085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-06 14:56:35.178916: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-06 14:56:35.180124: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-06 14:56:35.414208: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-06 14:56:36.640152: W tensorflow/compiler/tf

TensorFlow Version: 2.16.2
Input Shape: (256, 256, 3)
Anzahl Klassen: 10


In [2]:
# --- Experiment-Schalter ---
USE_KFOLD   = False   # True = 5-fache Kreuzvalidierung | False = einfacher Train/Val-Split
LOAD_MODELS = False   # True = gespeicherte Checkpoints laden statt neu trainieren

# --- Fine-Tuning Schalter ---
SKIP_PHASE_1  = True    # True = Lade die fertig trainierten Heads von der Festplatte!
DO_FINETUNING = True    # True = Phase 2 starten
UNFREEZE_ALL  = False   # False = Nur letzte Schichten entfrieren (Ideal für InceptionV3!)

# --- Datenpfade ---
DATA_PATH = './../data/Galaxy10_DECals.h5'
LOGS_DIR  = './../logs/'

# --- Hyperparameter ---
BATCH_SIZE     = 16
EPOCHS         = 100
WARMUP_EPOCHS     = 10      
FINETUNE_EPOCHS   = 50
LEARNING_RATE  = 1e-4
FINETUNE_LR    = 1e-5
K_FOLDS        = 5      # nur relevant wenn USE_KFOLD = True
RANDOM_STATE   = 42

In [3]:
# Reproduzierbarkeit
set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# GPU-Speicher nur nach Bedarf allozieren, nicht alles auf einmal reservieren
for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory Growth aktiviert: {gpu.name}")
    except RuntimeError as e:
        print(e)

print(f"TensorFlow {tf.__version__} | GPUs: {len(tf.config.list_physical_devices('GPU'))}")

Memory Growth aktiviert: /physical_device:GPU:0
TensorFlow 2.16.2 | GPUs: 1


2026-06-06 14:56:39.805511: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:56:40.096441: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:56:40.096515: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.


## 1. Daten laden und Splits vorbereiten
Wir laden die Bilder vollständig in den RAM (als `uint8`) und erzeugen die Indizes für unser K-Fold Cross-Validation Setup (inklusive separatem Test-Set).

In [4]:
print("Lade Datensatz in den RAM...")
images_raw, labels = load_data(DATA_PATH)

print("Erstelle Train/Val Folds und Test-Split...")
folds, X_test, y_test = prepare_splits(
    labels,
    use_kfold    = USE_KFOLD,
    k            = K_FOLDS,
    random_state = RANDOM_STATE,
)

print(f"Anzahl Folds: {len(folds)}")
print(f"Bilder im Test-Set: {len(X_test)}")

Lade Datensatz in den RAM...
Dataset geladen: 17,736 Bilder, Shape (17736, 256, 256, 3)
Erstelle Train/Val Folds und Test-Split...

Split-Modus : Einfacher Train/Val-Split
Train + Val : 14,188 Samples
Test        : 3,548 Samples
  Train 11,350 | Val 2,838
Anzahl Folds: 1
Bilder im Test-Set: 3548


## 2. Transfer Learning Helfer & Model Factories

Da unsere `run_training` Funktion für jeden Fold ein völlig **unbelastetes**, frisches Modell benötigt, definieren wir eine Basis-Funktion und sogenannte "Factories" (Funktionen, die bei Aufruf ein neues Modell zurückgeben).

In [5]:
def build_transfer_model(base_model, model_name, learning_rate=1e-4):
    """
    Kapselt das eingefrorene Basis-Modell und setzt einen neuen Kopf auf.
    """
    # Gewichte des vortrainierten Modells einfrieren
    base_model.trainable = False 
    
    inputs = Input(shape=INPUT_SHAPE)
    
    # training=False sorgt dafür, dass BatchNormalization im Inference-Modus bleibt
    x = base_model(inputs, training=False)
    
    # Klassifizierungs-Kopf (Top)
    x = Flatten(name="flatten")(x)  # Alternative zu GAP, wenn du mehr Kapazität möchtest
    # x = GlobalAveragePooling2D(name="gap")(x)
    # x = Dense(256, activation='relu', name="dense_256")(x)
    # x = BatchNormalization(name="bn_1")(x)
    # x = Dropout(0.5, name="dropout_1")(x)
    outputs = Dense(NUM_CLASSES, activation='softmax', name="predictions")(x)
    
    model = Model(inputs=inputs, outputs=outputs, name=model_name)
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# --- Model Factories ---

def resnet50_factory():
    base = ResNet50(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    return build_transfer_model(base, "ResNet50_Transfer", learning_rate=LEARNING_RATE)

def resnet152_factory():
    base = ResNet152(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    return build_transfer_model(base, "ResNet152_Transfer", learning_rate=LEARNING_RATE)

def inception_factory():
    base = InceptionV3(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    return build_transfer_model(base, "InceptionV3_Transfer", learning_rate=LEARNING_RATE)

def zoobot_factory():
    # EfficientNetB0 ist die Basis der meisten offiziellen Zoobot-Modelle
    base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=INPUT_SHAPE)
    return build_transfer_model(base, "Zoobot_EfficientNetB0_Transfer", learning_rate=LEARNING_RATE)

## 3. Training der Modelle

Wir iterieren über unsere definierten Factories und lassen jedes Modell durch unsere `run_training`-Schleife laufen.  
*Hinweis: Da die Modelle recht groß sind, setzen wir die `batch_size` hier konservativ auf 16, um OOM (Out of Memory) Fehler auf der GPU zu vermeiden.*

In [6]:
# Alle Modelle, die wir testen wollen
models_to_train = {
    "ResNet50": resnet50_factory,
    "ResNet152": resnet152_factory,
    "InceptionV3": inception_factory,
    # "Zoobot_EfficientNetB0": zoobot_factory
}

# Hier speichern wir die Ergebnisse für die spätere Evaluierung
training_results = {}

for model_name, factory in models_to_train.items():
    factory().summary()  # Zeige die Architektur des Modells an

for model_name, factory in models_to_train.items():
    print(f"\n{'=' * 80}")
    print(f"STARTE TRAINING FÜR: {model_name}")
    print(f"{'=' * 80}")

    factory().summary()  # Zeige die Architektur des Modells an
    
    log_dir = f"{LOGS_DIR}/transfer_learning/{model_name.lower()}_flatten_dense_v2"
    
    if SKIP_PHASE_1:
        print(f"ÜBERSPRINGE PHASE 1: Lade fertige '{model_name}' Modelle von der Festplatte...")
        fold_models = []
        
        # Wir suchen alle gespeicherten Modelle in diesem log_dir
        # Passe die Endung an, falls deine train.py ".h5" statt ".keras" speichert!
        model_paths = sorted(glob.glob(f"{log_dir}/fold_*.keras") + glob.glob(f"{log_dir}/fold_*.h5"))
        
        if not model_paths:
             print(f"WARNUNG: Keine Modelle in {log_dir} gefunden! Überspringe dieses Modell.")
             continue
             
        for path in model_paths:
            print(f" -> Lade {path}")
            model = tf.keras.models.load_model(path)
            fold_models.append(model)
            
        # Wir setzen histories auf leere Listen, da wir sie nicht plotten müssen, 
        # wenn wir direkt ins Fine-Tuning springen.
        fold_histories = [None] * len(fold_models)
        
    else:
        print(f"STARTE PHASE 1 (WARM-UP) FÜR: {model_name}")
        factory().summary()
        
        fold_models, fold_histories = run_training(
            folds=folds,
            images_raw=images_raw,
            labels=labels,
            logs_dir=log_dir,
            model_factory=factory,
            batch_size=BATCH_SIZE,
            epochs=WARMUP_EPOCHS,
            early_stopping_patience=5,
            reduce_lr_patience=2,
            load_models=LOAD_MODELS  
        )
    
    training_results[model_name] = {
        "models": fold_models,
        "histories": fold_histories,
        "log_dir": log_dir
    }

2026-06-06 14:57:23.526651: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:57:23.526798: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:57:23.526877: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:57:23.761212: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:2d:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-06-06 14:57:23.761294: I external/local_xla/xla/stream_executor

Model: "ResNet50_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 8, 8, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 131072)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │     1,310,730 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,898,442 (94.98 MB)

 Trainable params: 1,310,730 (5.00 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 6, 6, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73728)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │       737,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,540,074 (85.98 MB)

 Trainable params: 737,290 (2.81 MB)

 Non-trainable params: 21,802,784 (83.17 MB)

ÜBERSPRINGE PHASE 1: Lade fertige 'InceptionV3' Modelle von der Festplatte...
WARNUNG: Keine Modelle in ./../logs//transfer_learning/inceptionv3_flatten_dense_v2 gefunden! Überspringe dieses Modell.


## 4. Evaluierung & Ergebnisse

Jetzt, da alle Architekturen trainiert sind (bzw. die Checkpoints geladen wurden), nutzen wir unsere `evaluate.py` Methoden, um die Accuracy auf dem gemeinsamen Test-Set zu ermitteln und die Ergebnisse zu plotten.

In [9]:
for model_name, results in training_results.items():
    print(f"\n\n{'*' * 80}")
    print(f"EVALUIERUNG FÜR: {model_name}")
    print(f"{'*' * 80}")
    
    log_dir = results["log_dir"]
    fold_models = results["models"]
    fold_histories = results["histories"]
    
    # 1. Plot der Trainings-Historie
    plot_training_history(fold_histories, save_dir=log_dir)
    
    # 2. Evaluation auf dem unseen Test-Set
    test_accuracies, best_model, best_fold_idx = evaluate_models(
        fold_models=fold_models,
        images_raw=images_raw,
        labels=labels,
        X_test=X_test,
        batch_size=BATCH_SIZE
    )
    
    # 3. Confusion Matrix des besten Folds
    cm_path = os.path.join(log_dir, "confusion_matrix.png")
    plot_confusion_matrix(
        best_model=best_model,
        images_raw=images_raw,
        labels=labels,
        X_test=X_test,
        best_fold_idx=best_fold_idx,
        batch_size=BATCH_SIZE,
        save_path=cm_path
    )

## 5. (Optional) Fine-Tuning des besten Modells

Wenn eines der Modelle besonders gut abgeschnitten hat, kannst du hier die Schichten des vortrainierten Backbones auftauen und mit einer sehr kleinen Lernrate (`1e-5`) für ein paar weitere Epochen trainieren (Fine-Tuning).

In [8]:
if DO_FINETUNING:
    print(f"\n{'*' * 80}")
    print(f"STARTE PHASE 2: FINE-TUNING")
    print(f"{'*' * 80}")
    
    for model_name, results in training_results.items():
        print(f"\n--- Fine-Tuning für Architektur: {model_name} ---")
        fold_models = results["models"]
        
        for fold_idx, model in enumerate(fold_models):
            print(f"Bearbeite Fold {fold_idx + 1}/{len(fold_models)}...")
            
            # 1. Das Basis-Modell im Keras-Graphen finden
            base_model = None
            for layer in model.layers:
                if isinstance(layer, Model):
                    base_model = layer
                    break
                    
            if base_model is None:
                print("Fehler: Konnte Basis-Modell nicht finden.")
                continue
                
            # 2. Backbone-Schichten gezielt entfrieren
            base_model.trainable = True
            
            if not UNFREEZE_ALL:
                print(" -> Modus: PARTIELL (Nur die obersten Schichten entfrieren)")
                for layer in base_model.layers:
                    layer.trainable = False # Standardmäßig alles einfrieren
                    
                    # --- Architekturspezifisches Entfrieren ---
                    arch_name = base_model.name.lower()
                    if "inception" in arch_name:
                        # InceptionV3: Nur die letzten Mixed-Blöcke
                        if layer.name.startswith("mixed9") or layer.name.startswith("mixed10"):
                            layer.trainable = True
                            
                    elif "resnet" in arch_name:
                        # ResNet: Nur den letzten Conv-Block
                        if layer.name.startswith("conv5"):
                            layer.trainable = True
                            
                    elif "efficientnet" in arch_name:
                        # EfficientNet: Nur den obersten Block
                        if layer.name.startswith("block7") or layer.name.startswith("top"):
                            layer.trainable = True
            else:
                print(" -> Modus: KOMPLETT (Ganzes Backbone wird trainiert)")

            # Info: Batch Normalization bleibt sicher!
            # Da base_model(inputs, training=False) in der Factory gesetzt wurde, 
            # ändert Keras die BN-Gewichte nicht, selbst wenn trainable=True ist.
            
            # 3. Neu kompilieren mit SEHR kleiner Lernrate
            model.compile(
                optimizer=Adam(learning_rate=FINETUNE_LR), 
                loss="categorical_crossentropy", 
                metrics=["accuracy"]
            )
            
            # 4. Train/Val Indizes für diesen Fold laden
            train_idx, val_idx = folds[fold_idx]
            X_train_fold, y_train_fold = images_raw[train_idx], labels[train_idx]
            X_val_fold, y_val_fold = images_raw[val_idx], labels[val_idx]
            
            # 5. Callbacks für sauberes Training definieren
            callbacks = [
                EarlyStopping(patience=5, restore_best_weights=True, monitor="val_loss"),
                ReduceLROnPlateau(patience=2, factor=0.5, min_lr=1e-7, monitor="val_loss")
            ]
            
            # 6. Fine-Tuning direkt über model.fit() ausführen
            print(f" -> Starte Training mit Lernrate {FINETUNE_LR}...")
            ft_history = model.fit(
                X_train_fold, y_train_fold,
                validation_data=(X_val_fold, y_val_fold),
                epochs=FINETUNE_EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=callbacks,
                verbose=1
            )
            
            if results["histories"][fold_idx] is not None:
                for key in results["histories"][fold_idx].history.keys():
                    results["histories"][fold_idx].history[key].extend(ft_history.history[key])
            else:
                # Wenn wir Phase 1 übersprungen haben, ist die FT-History jetzt unsere einzige History
                results["histories"][fold_idx] = ft_history
                
            # WICHTIG: Das Fine-Tuning-Modell speichern, damit es für Zelle 12 verfügbar ist!
            ft_save_path = f"{results['log_dir']}/fold_{fold_idx + 1}_finetuned.keras"
            model.save(ft_save_path)
            print(f" -> Fine-Tuned Modell gespeichert unter: {ft_save_path}")
                
    print("\nFine-Tuning abgeschlossen! Du kannst nun die Evaluierungs-Zelle (Zelle 12) erneut ausführen, um die Plots mit den neuen Gewichten zu generieren.")


********************************************************************************
STARTE PHASE 2: FINE-TUNING
********************************************************************************

Fine-Tuning abgeschlossen! Du kannst nun die Evaluierungs-Zelle (Zelle 12) erneut ausführen, um die Plots mit den neuen Gewichten zu generieren.
